In [ ]:
import torch
import pprint
import numpy as np

from tqdm import tqdm
from src.util.celano_lab_scripts import process_image
from src.datasets.mos2_sef import MOS2SEFDataset, Formulation
from src.models.our_method.swin_cafm import SwinCAFM
from src.models.our_method.older_surrogate import MultiHeadOlderSurrogate

# p(y_char | y_hat)
surrogate_weights = "/playpen/mufan/levi/tianlong-chen-lab/material-super-resolution/__exps__/y-task-formulations/p(y | y_sparse)/e. surrogate standalone train-runs/2025-02-09_14-31-35_older_surrogate_mh-ViT-L16-l[-2]+layernorm-adamW-lr=1e-4-augs=True/older_surrogate_mh-ViT-L16-l[-2]+layernorm-adamW-lr=1e-4-augs=True_best_older_surrogate.pth"
surrogate = torch.load(surrogate_weights).cuda().float()

# p(y | y_sparse)
model_weights = "/playpen/mufan/levi/tianlong-chen-lab/material-super-resolution/__exps__/y-task-formulations/p(y | y_sparse)/a. train-runs/2025-02-10/2025-02-10_10-53-35_swinir-loss=OLDER-Perceptual(raw-out, y)/swinir-loss=OLDER-Perceptual(raw-out, y)_best.pth"
model = torch.load(model_weights).cuda().float()

model.train()
surrogate.eval()

formulation = Formulation.P_Y_BAR_Y_SPARSE
train_dataset = MOS2SEFDataset("train", formulation, 128, 1, 750)
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=1, shuffle=False, num_workers=0)
val_dataset = MOS2SEFDataset("val", formulation, 128, 1, 250)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=0)

optimizer = torch.optim.Adam(model.parameters())

IMG_SIZE_UM=2.0
values = {}

Xs = []; ys = []

@torch.no_grad()
def sample():
    
    for i, data in tqdm(enumerate(train_dataloader), total=len(train_dataloader)):
        
        y = data['y'].cuda().float()
        mask = data['mask'].cuda().float()
        y_sparse = (mask * y).cuda().float()
        y_hat = model(y_sparse).cuda().float()
        
        # older_pred = surrogate(y/y_hat)
        char_gt = surrogate(y); char_pred = surrogate(y_hat)
        l1 = torch.nn.functional.l1_loss(y, y_hat)
        
        X_arr = char_pred.detach().cpu().numpy()
        y_arr = l1.detach().cpu().numpy()
        
        Xs.append(X_arr)
        ys.append(y_arr)
                      
sample()

/tmp/ipykernel_1456110/3001330644.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  surrogate = torch.load(surrogate_weights).cuda().float()
/tmp/ipykernel_1456110/300133

# [Training Set] OLDER-L1 Feature Correlation
---

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# -------------------------------------------------------
# 1. Pretend these are your collected surrogate features & L1 losses
#    X.shape => (N, 9)
#    y.shape => (N,)
# -------------------------------------------------------
# For illustration, let's assume you already have:
# X = np.array(...)  # Nx9 surrogate outputs
# y = np.array(...)  # Nx1 or Nx, L1 losses

# Xs = np.array(Xs).squeeze()
# ys = np.array(ys).squeeze()

X = Xs
y = ys

# -------------------------------------------------------
# 2. Split into train/test to evaluate model performance
# -------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# -------------------------------------------------------
# 3. Train a simple Random Forest Regressor
# -------------------------------------------------------
rf_model = RandomForestRegressor(n_estimators=100, random_state=40)
rf_model.fit(X_train, y_train)

# -------------------------------------------------------
# 4. Evaluate performance on the test set
# -------------------------------------------------------
y_pred = rf_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"Test MAE: {mae:.4f}")

# -------------------------------------------------------
# 5. Inspect Feature Importances
# -------------------------------------------------------
importances = rf_model.feature_importances_
for idx, imp in enumerate(importances):
    print(f"Feature {idx+1}: Importance = {imp:.4f}")

# Features with higher importance are more influential
# to the model's predicted L1 in this simplified setting.

Test MAE: 0.0011
Feature 1: Importance = 0.7410
Feature 2: Importance = 0.0768
Feature 3: Importance = 0.0118
Feature 4: Importance = 0.0156
Feature 5: Importance = 0.0543
Feature 6: Importance = 0.0195
Feature 7: Importance = 0.0193
Feature 8: Importance = 0.0299
Feature 9: Importance = 0.0318


# [Validation Set] OLDER-L1 Feature Correlation
---

In [ ]:
Xs = []; ys = []
@torch.no_grad()

def sample():
    for i, data in tqdm(enumerate(val_dataloader), total=len(val_dataloader)):
        y = data['y'].cuda().float()
        mask = data['mask'].cuda().float()
        y_sparse = (mask * y).cuda().float()
        y_hat = model(y_sparse).cuda().float()
        # older_pred = surrogate(y/y_hat)
        char_gt = surrogate(y); char_pred = surrogate(y_hat)
        l1 = torch.nn.functional.l1_loss(y, y_hat)        
        X_arr = char_pred.detach().cpu().numpy()
        y_arr = l1.detach().cpu().numpy()
        Xs.append(X_arr)
        ys.append(y_arr)
                      
sample()

100%|██████████| 250/250 [00:25<00:00,  9.84it/s]


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# -------------------------------------------------------
# 1. Pretend these are your collected surrogate features & L1 losses
#    X.shape => (N, 9)
#    y.shape => (N,)
# -------------------------------------------------------
# For illustration, let's assume you already have:
# X = np.array(...)  # Nx9 surrogate outputs
# y = np.array(...)  # Nx1 or Nx, L1 losses

Xs = np.array(Xs).squeeze()
ys = np.array(ys).squeeze()

X = Xs
y = ys

# -------------------------------------------------------
# 2. Split into train/test to evaluate model performance
# -------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# -------------------------------------------------------
# 3. Train a simple Random Forest Regressor
# -------------------------------------------------------
rf_model = RandomForestRegressor(n_estimators=100, random_state=20)
rf_model.fit(X_train, y_train)

# -------------------------------------------------------
# 4. Evaluate performance on the test set
# -------------------------------------------------------
y_pred = rf_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"Test MAE: {mae:.4f}")

# -------------------------------------------------------
# 5. Inspect Feature Importances
# -------------------------------------------------------
importances = rf_model.feature_importances_
for idx, imp in enumerate(importances):
    print(f"Feature {idx+1}: Importance = {imp:.4f}")

# Features with higher importance are more influential
# to the model's predicted L1 in this simplified setting.

Test MAE: 0.0011
Feature 1: Importance = 0.7479
Feature 2: Importance = 0.0696
Feature 3: Importance = 0.0167
Feature 4: Importance = 0.0148
Feature 5: Importance = 0.0510
Feature 6: Importance = 0.0168
Feature 7: Importance = 0.0233
Feature 8: Importance = 0.0232
Feature 9: Importance = 0.0367
